In [1]:
import sys
import sympy as sp
import numpy as np
import scipy
from scipy.integrate import solve_ivp
from IPython.display import display
sp.init_printing(use_latex='mathjax')
print('Python:', sys.version.split()[0], '| SymPy:', sp.__version__, '| NumPy:', np.__version__, '| SciPy:', scipy.__version__)
def zero(expr):
    assert sp.simplify(sp.trigsimp(expr)) == 0, expr


Python: 3.12.14 | SymPy: 1.14.0 | NumPy: 2.5.3 | SciPy: 1.18.1


## Derivation 1 — internal force and torque
Define $F_{12}$ as the force on 1 due to 2. The given center-of-mass equation forces $F_{12}+F_{21}=0$. Substituting this into the angular-momentum balance leaves $(r_1-r_2)\times F_{12}=0$. For distinct particles, the nullspace of this cross-product operator is exactly the line of separation.

Angular momentum is about a fixed inertial origin. Constant masses and ordinary particle momentum $m v$ are assumed.

In [2]:
r1=sp.Matrix(sp.symbols('x1 y1 z1', real=True))
r2=sp.Matrix(sp.symbols('x2 y2 z2', real=True))
f12=sp.Matrix(sp.symbols('Fx Fy Fz', real=True))
f21=sp.Matrix(sp.symbols('Gx Gy Gz', real=True))
weak=sp.solve(list(f12+f21), list(f21), dict=True)[0]
assert (f21.subs(weak)+f12)==sp.zeros(3,1)
torque=(r1.cross(f12)+r2.cross(f21)).subs(weak)
assert sp.simplify(torque-(r1-r2).cross(f12))==sp.zeros(3,1)
d=r1-r2
C=sp.Matrix([[0,-d[2],d[1]],[d[2],0,-d[0]],[-d[1],d[0],0]])
assert sp.simplify(C*d)==sp.zeros(3,1)
# A generic nonzero separation has rank 2 and a one-dimensional nullspace.
assert C.rank()==2
print('Weak law:'); display(weak)
print('Residual internal torque:'); display(torque)
print('Cross-product matrix rank:', C.rank())
# Numerical central-force example, and a noncentral counterexample to weak => strong.
dn=np.array([1.,2.,3.]); fn=2.5*dn
assert np.allclose(np.cross(dn,fn),0)
noncentral=np.array([0.,1.,0.])
assert np.linalg.norm(np.cross(dn,noncentral))>0
print('Equal/opposite alone does not imply zero internal torque:', np.cross(dn,noncentral))

Weak law:
Residual internal torque:
Cross-product matrix rank: 2
Equal/opposite alone does not imply zero internal torque: [-3.  0.  1.]


{Gx: -Fx, Gy: -Fy, Gz: -Fz}

⎡-Fy⋅z₁ + Fy⋅z₂ + Fz⋅y₁ - Fz⋅y₂⎤
⎢                              ⎥
⎢Fx⋅z₁ - Fx⋅z₂ - Fz⋅x₁ + Fz⋅x₂ ⎥
⎢                              ⎥
⎣-Fx⋅y₁ + Fx⋅y₂ + Fy⋅x₁ - Fy⋅x₂⎦

## Derivation 2 — Kepler orbits
Use $V=-k/r$, $k>0$, and $J\ne0$. Circular motion extremizes $U_{\mathrm{eff}}$; a parabola has zero total energy, and perihelion has zero radial velocity. Part (a) compares equal angular momentum; part (b) compares equal radius, a different condition.

In [3]:
r, mu, k, J=sp.symbols('r mu k J', positive=True)
Ueff=J**2/(2*mu*r**2)-k/r
rc=J**2/(mu*k); rp=J**2/(2*mu*k)
zero(sp.diff(Ueff,r).subs(r,rc))
zero(Ueff.subs(r,rp))
zero(rp/rc-sp.Rational(1,2))
vp=sp.sqrt(2*k/(mu*r)); vc=sp.sqrt(k/(mu*r))
zero(vp/vc-sp.sqrt(2))
print('rc, rp, rp/rc, vp/vc:'); display(rc,rp,sp.simplify(rp/rc),sp.simplify(vp/vc))

rc, rp, rp/rc, vp/vc:


 2 
J  
───
k⋅μ

  2  
 J   
─────
2⋅k⋅μ

1/2

√2

In [4]:
# Integrate Newton's Cartesian equations independently of the conic formula.
# Units chosen so mu=k=J=1: circular radius=1, parabolic perihelion=0.5.
def kepler(t,y):
    q=y[:2]; v=y[2:]; R=np.linalg.norm(q)
    return np.r_[v,-q/R**3]
times=np.linspace(0,3,601)
par=solve_ivp(kepler,[0,3],[0.5,0,0,2],t_eval=times,rtol=1e-11,atol=1e-12)
cir=solve_ivp(kepler,[0,3],[1,0,0,1],t_eval=times,rtol=1e-11,atol=1e-12)
assert par.success and cir.success
R=np.linalg.norm(par.y[:2],axis=0); speed=np.linalg.norm(par.y[2:],axis=0)
Jnum=par.y[0]*par.y[3]-par.y[1]*par.y[2]
E=speed**2/2-1/R
ratio=speed/np.sqrt(1/R)
# The parabola also obeys r+x=p=J^2/(mu*k)=1.
checks={'parabolic energy error':np.max(np.abs(E)),
        'angular momentum error':np.max(np.abs(Jnum-1)),
        'speed ratio error':np.max(np.abs(ratio-np.sqrt(2))),
        'conic equation error':np.max(np.abs(R+par.y[0]-1)),
        'circular radius error':np.max(np.abs(np.linalg.norm(cir.y[:2],axis=0)-1))}
for name,error in checks.items():
    print(f'{name}: {error:.3e}')
    assert error<1e-8
assert np.isclose(R[0]/np.linalg.norm(cir.y[:2,0]),0.5)


parabolic energy error: 3.277e-11
angular momentum error: 3.077e-11
speed ratio error: 1.955e-11
conic equation error: 6.991e-12
circular radius error: 8.757e-12


## Problem 1(a) — derive from the Cartesian geometry
Positive $z$ points downward. On the pictured linkage branch, $z_2=2l\cos\theta$, while $\phi=\Omega t$ is imposed by the drive. Do **not** conserve axial angular momentum: the drive can exert torque. Both masses contribute to the kinetic energy; omitting the bead loses the $4\sin^2\theta$ factor.

The rod constraint also permits a degenerate branch $z_2=0$ with coincident axial joints. We retain the continuous, nondegenerate branch shown in the exam.

In [5]:
t=sp.symbols('t', real=True)
m,l,g=sp.symbols('m l g', positive=True)
Om=sp.symbols('Omega', real=True)
q=sp.Function('theta')(t)
R1=sp.Matrix([l*sp.sin(q)*sp.cos(Om*t),l*sp.sin(q)*sp.sin(Om*t),l*sp.cos(q)])
R2=sp.Matrix([0,0,2*l*sp.cos(q)])
zero(R1.dot(R1)-l**2)
zero((R2-R1).dot(R2-R1)-l**2)
v1=sp.simplify(sp.diff(R1,t).dot(sp.diff(R1,t)))
v2=sp.simplify(sp.diff(R2,t).dot(sp.diff(R2,t)))
T=sp.trigsimp(m*(v1+v2)/2)
V=-m*g*(R1[2]+R2[2]); Lag=T-V
EL=sp.diff(sp.diff(Lag,sp.diff(q,t)),t)-sp.diff(Lag,q)
expected=(1+4*sp.sin(q)**2)*sp.diff(q,t,2)+4*sp.sin(q)*sp.cos(q)*sp.diff(q,t)**2+(3*g/l-Om**2*sp.cos(q))*sp.sin(q)
zero(EL/(m*l**2)-expected)
print('v1 squared, v2 squared, T, V, normalized Euler-Lagrange residual:')
display(v1,v2,T,V,sp.simplify(EL/(m*l**2)))
print('Geometry and Euler-Lagrange checks passed.')

v1 squared, v2 squared, T, V, normalized Euler-Lagrange residual:
Geometry and Euler-Lagrange checks passed.


   ⎛                          2⎞
 2 ⎜ 2    2         ⎛d       ⎞ ⎟
l ⋅⎜Ω ⋅sin (θ(t)) + ⎜──(θ(t))⎟ ⎟
   ⎝                ⎝dt      ⎠ ⎠

                          2
   2    2       ⎛d       ⎞ 
4⋅l ⋅sin (θ(t))⋅⎜──(θ(t))⎟ 
                ⎝dt      ⎠ 

     ⎛                                       2             2⎞
 2   ⎜ 2    2              2       ⎛d       ⎞    ⎛d       ⎞ ⎟
l ⋅m⋅⎜Ω ⋅sin (θ(t)) + 4⋅sin (θ(t))⋅⎜──(θ(t))⎟  + ⎜──(θ(t))⎟ ⎟
     ⎝                             ⎝dt      ⎠    ⎝dt      ⎠ ⎠
─────────────────────────────────────────────────────────────
                              2                              

-3⋅g⋅l⋅m⋅cos(θ(t))

   2                                             2                             ↪
  Ω ⋅sin(2⋅θ(t))   3⋅g⋅sin(θ(t))        2       d                         ⎛d   ↪
- ────────────── + ───────────── + 4⋅sin (θ(t))⋅───(θ(t)) + 2⋅sin(2⋅θ(t))⋅⎜──( ↪
        2                l                        2                       ⎝dt  ↪
                                                dt                             ↪

↪       2    2       
↪      ⎞    d        
↪ θ(t))⎟  + ───(θ(t))
↪      ⎠      2      
↪           dt       

## Problem 1(b) — equilibrium, stability, and limits
The effective potential is $U=-3mgl\cos\theta-ml^2\Omega^2\sin^2\theta/2$. It belongs to the driven, reduced equation; it is not the gravitational potential alone.

Modulo azimuth, on $0\leq\theta\leq\pi$ the counts are 2 for $|\Omega|\leq\sqrt{3g/l}$ and 3 above it. If the bead is restricted below the origin ($0\leq\theta<\pi/2$), the counts are 1 and 2. The upper vertical equilibrium is unstable. At the threshold the tilted solution coincides with the downward one, so do not count it twice. Stability here refers to the allowed $\theta$ motion with prescribed rotation.

In [6]:
th=sp.symbols('theta', real=True)
U=-3*m*g*l*sp.cos(th)-m*l**2*Om**2*sp.sin(th)**2/2
Up=sp.diff(U,th); Upp=sp.diff(U,th,2)
zero(Up-m*l**2*sp.sin(th)*(3*g/l-Om**2*sp.cos(th)))
zero(Upp.subs(th,0)-m*l**2*(3*g/l-Om**2))
zero(Upp.subs(th,sp.pi)+m*l**2*(3*g/l+Om**2))
# At a nonvertical equilibrium Omega^2=3g/(l*cos(theta)).
zero((Upp-m*l**2*Om**2*sp.sin(th)**2).subs(Om**2,3*g/(l*sp.cos(th))))
threshold=sp.series((U-U.subs(th,0)).subs(Om**2,3*g/l),th,0,6)
assert threshold.removeO()==sp.Rational(3,8)*m*g*l*th**4
print('Threshold potential expansion:'); display(threshold)
# Small oscillation frequency squared on the tilted branch.
frequency2=Om**2*sp.sin(th)**2/(1+4*sp.sin(th)**2)
print('Tilted small-oscillation frequency squared:'); display(frequency2)
print('Dimensionless rotation s=Omega/sqrt(3g/l):')
for s in [0,0.5,1,1.01,2,10]:
    equilibria=[0.,np.pi]
    if s>1: equilibria.append(np.arccos(1/s**2))
    for angle in equilibria:
        assert abs(np.sin(angle)*(1-s**2*np.cos(angle)))<1e-10
    print(f's={s:5.2f}, angles (degrees)={np.round(np.degrees(equilibria),4)}, full count={len(equilibria)}, below-origin count={len(equilibria)-1}')
# Exact tilted coordinates and asymptotic expansion.
s=sp.symbols('s',positive=True)
zero(2*l*(3*g/(l*Om**2))-6*g/Om**2)
assert sp.limit(sp.acos(1/s**2),s,sp.oo)==sp.pi/2
print('Fast-rotation theta limit = pi/2; z1=3g/Omega^2 and z2=6g/Omega^2.')

Threshold potential expansion:
Tilted small-oscillation frequency squared:
Dimensionless rotation s=Omega/sqrt(3g/l):
s= 0.00, angles (degrees)=[  0. 180.], full count=2, below-origin count=1
s= 0.50, angles (degrees)=[  0. 180.], full count=2, below-origin count=1
s= 1.00, angles (degrees)=[  0. 180.], full count=2, below-origin count=1
s= 1.01, angles (degrees)=[  0.     180.      11.3928], full count=3, below-origin count=2
s= 2.00, angles (degrees)=[  0.     180.      75.5225], full count=3, below-origin count=2
s=10.00, angles (degrees)=[  0.    180.     89.427], full count=3, below-origin count=2
Fast-rotation theta limit = pi/2; z1=3g/Omega^2 and z2=6g/Omega^2.


         4        
3⋅g⋅l⋅m⋅θ     ⎛ 6⎞
────────── + O⎝θ ⎠
    8             

  2    2     
 Ω ⋅sin (θ)  
─────────────
     2       
4⋅sin (θ) + 1

### Numerical integration of the rod equation
Set $m=l=g=1$, $\Omega=2>\sqrt3$, and perturb the stable tilted state. Check the reduced conserved energy $H_{\rm red}=\tfrac12 ml^2(1+4\sin^2\theta)\dot\theta^2+U$. This is a Jacobi-type integral for the prescribed rotation, **not** the laboratory mechanical energy $T+V$. Also compare the oscillation period with the independent curvature prediction.

In [7]:
omega=2.0; equilibrium=np.arccos(3/omega**2)
wn=np.sqrt(omega**2*np.sin(equilibrium)**2/(1+4*np.sin(equilibrium)**2))
period=2*np.pi/wn
amplitude=1e-4
def rods(t,y):
    angle,velocity=y; sn=np.sin(angle); cs=np.cos(angle)
    acceleration=-(4*sn*cs*velocity**2+(3-omega**2*cs)*sn)/(1+4*sn**2)
    return [velocity,acceleration]
def crossing(t,y): return y[0]-equilibrium
crossing.direction=-1
sol=solve_ivp(rods,[0,5*period],[equilibrium+amplitude,0],events=crossing,
              rtol=1e-11,atol=1e-13,max_step=period/100)
assert sol.success
angle,vel=sol.y
H=0.5*(1+4*np.sin(angle)**2)*vel**2-3*np.cos(angle)-0.5*omega**2*np.sin(angle)**2
energy_drift=np.max(np.abs(H-H[0]))
measured=np.mean(np.diff(sol.t_events[0]))
period_error=abs(measured/period-1)
assert energy_drift<1e-9
assert period_error<1e-5
assert np.max(np.abs(angle-equilibrium))<1.01*amplitude
print(f'Reduced energy drift: {energy_drift:.3e}')
print(f'Predicted period: {period:.10f}; measured: {measured:.10f}')
print(f'Relative period error: {period_error:.3e}')
print('ALL SYMBOLIC AND NUMERICAL ASSERTIONS PASSED.')

Reduced energy drift: 8.882e-16
Predicted period: 7.8763896157; measured: 7.8763896561
Relative period error: 5.131e-09
ALL SYMBOLIC AND NUMERICAL ASSERTIONS PASSED.
